
# 05 — Tanore ↔ Manda Bidirectional Cross-Area Transfer Test

এই একটি notebook-এই দুইটি true geographic transfer experiment চলবে:

1. **Tanore training → Manda independent validation**
2. **Manda training → Tanore independent validation**

প্রতিটি direction-এ আবার দুইটি data stream এবং দুইটি classifier পরীক্ষা হবে:

- Fused-Hybrid + Random Forest
- Fused-Hybrid + XGBoost
- Planet-only + Random Forest
- Planet-only + XGBoost

অর্থাৎ মোট **8টি transfer experiment**।

## Scientific rule
Target area-এর training sample দিয়ে **কোনো tuning, threshold selection বা model fitting করা হবে না**।  
Hyperparameter tuning এবং probability threshold **শুধু source-area training data** দিয়ে spatial/grouped CV-এর মাধ্যমে নির্ধারণ করা হবে। তারপর final model সরাসরি target-area independent validation sample-এ test হবে।

## Before Run All
এই দুই classification notebook আগে complete থাকতে হবে:

- `03_Tanore_Q1_Classification_VSCode.ipynb`
- `03_Manda_Q1_Classification_VSCode.ipynb`

এবং দুই area-এর `Classification_Q1/tables` folder-এ training/validation CSV থাকতে হবে।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:

# CELL 1 — Imports, paths, and settings

from pathlib import Path
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    os.environ.get(
        "BORO_PROJECT_ROOT",
        str(Path.cwd()),
    )
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "Outputs"
    / "Q1_Extensions"
    / "Cross_Area_Transfer"
)
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"

for folder in [OUTPUT_ROOT, FIGURE_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
CV_SPLITS = 5

# Publication run
RF_SEARCH_ITERATIONS = 15
XGB_SEARCH_ITERATIONS = 20
BOOTSTRAP_REPLICATES = 1000

# True = quick diagnostic only; False = final publication run
QUICK_MODE = False

if QUICK_MODE:
    RF_SEARCH_ITERATIONS = 3
    XGB_SEARCH_ITERATIONS = 3
    BOOTSTRAP_REPLICATES = 200

AREAS = ["Tanore", "Manda"]
STREAMS = ["FusedHybrid", "PlanetOnly"]
MODELS = ["RandomForest", "XGBoost"]

print("Project root :", PROJECT_ROOT)
print("Output folder:", OUTPUT_ROOT)
print("Quick mode   :", QUICK_MODE)


In [ ]:

# CELL 2 — Exact predictor schema used in the classification notebooks

DATES = ["Jan", "Mar", "Apr1", "Apr2"]
BAND_NAMES = ["Blue", "Green", "Red", "NIR"]

BASE_FEATURE_NAMES = [
    feature
    for date_name in DATES
    for feature in [
        *(f"{date_name}_{band}" for band in BAND_NAMES),
        f"{date_name}_NDVI",
    ]
]

DERIVED_FEATURE_NAMES = [
    "NDVI_mean",
    "NDVI_std",
    "NDVI_min",
    "NDVI_max",
    "NDVI_amplitude",
    "dNDVI_Mar_Jan",
    "dNDVI_Apr1_Mar",
    "dNDVI_Apr2_Apr1",
    "dNDVI_Apr1_Jan",
    "NDVI_peak_timing",
]

FEATURE_NAMES = BASE_FEATURE_NAMES + DERIVED_FEATURE_NAMES

print("Number of predictors:", len(FEATURE_NAMES))
print(FEATURE_NAMES)


In [ ]:

# CELL 3 — Load and validate source/target sample tables

def table_path(area, stream, split):
    return (
        PROJECT_ROOT
        / "Outputs"
        / area
        / "Classification_Q1"
        / "tables"
        / f"Q1_{stream}_{split}_Samples.csv"
    )


def load_table(area, stream, split):
    path = table_path(area, stream, split)

    if not path.exists():
        raise FileNotFoundError(
            "\nRequired file was not found:\n"
            f"{path}\n\n"
            f"Run the final 03_{area}_Q1_Classification notebook first."
        )

    df = pd.read_csv(path)

    required = {
        "sample_id",
        "class",
        "group",
        *FEATURE_NAMES,
    }

    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(
            f"{path.name} is missing these required columns:\n{missing}"
        )

    if df[FEATURE_NAMES].isna().any().any():
        bad_rows = int(df[FEATURE_NAMES].isna().any(axis=1).sum())
        raise ValueError(
            f"{path.name} contains {bad_rows} rows with missing predictor values."
        )

    unique_classes = sorted(df["class"].dropna().unique().tolist())
    if set(unique_classes) != {0, 1}:
        raise ValueError(
            f"{path.name}: class must contain both 0 and 1. Found {unique_classes}"
        )

    return df.copy()


sample_tables = {
    area: {
        stream: {
            split: load_table(area, stream, split)
            for split in ["Training", "Validation"]
        }
        for stream in STREAMS
    }
    for area in AREAS
}

summary_rows = []

for area in AREAS:
    for stream in STREAMS:
        for split in ["Training", "Validation"]:
            df = sample_tables[area][stream][split]
            summary_rows.append({
                "area": area,
                "stream": stream,
                "split": split,
                "n_samples": len(df),
                "n_rice": int((df["class"] == 1).sum()),
                "n_nonrice": int((df["class"] == 0).sum()),
                "n_groups": int(df["group"].astype(str).nunique()),
            })

input_summary = pd.DataFrame(summary_rows)
display(input_summary)

print("\n✅ All required classification sample tables were found.")


In [ ]:

# CELL 4 — Source-only spatial CV, model tuning, and evaluation helpers

def make_spatial_cv(y, groups):
    group_table = (
        pd.DataFrame({
            "group": groups,
            "class": y,
        })
        .drop_duplicates()
    )

    class_group_counts = (
        group_table
        .groupby("class")["group"]
        .nunique()
    )

    n_splits = min(
        CV_SPLITS,
        int(class_group_counts.min()),
        int(group_table["group"].nunique()),
    )

    if n_splits < 3:
        raise ValueError(
            "At least 3 independent groups per class are required "
            "for the source-area spatial CV."
        )

    return StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED,
    )


def optimal_f1_threshold(y_true, probability):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probability,
    )

    if thresholds.size == 0:
        return 0.5

    f1_values = (
        2.0
        * precision[:-1]
        * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12,
        )
    )

    return float(
        thresholds[
            int(np.nanargmax(f1_values))
        ]
    )


def build_search(model_name, y):
    if model_name == "RandomForest":
        estimator = RandomForestClassifier(
            random_state=RANDOM_SEED,
            class_weight="balanced",
            n_jobs=1,
        )

        parameters = {
            "n_estimators": [300, 500, 800, 1000],
            "max_depth": [None, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4, 8],
            "max_features": ["sqrt", "log2", 0.4, 0.7],
            "bootstrap": [True, False],
        }

        if QUICK_MODE:
            parameters = {
                "n_estimators": [80, 120, 180],
                "max_depth": [None, 10, 15],
                "min_samples_split": [2, 5],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", 0.7],
                "bootstrap": [True, False],
            }

        n_iter = RF_SEARCH_ITERATIONS

    elif model_name == "XGBoost":
        negative = max(int((y == 0).sum()), 1)
        positive = max(int((y == 1).sum()), 1)

        estimator = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=1,
            scale_pos_weight=negative / positive,
        )

        parameters = {
            "n_estimators": [300, 500, 700, 1000],
            "max_depth": [3, 4, 5, 6, 8],
            "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
            "subsample": [0.65, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.75, 0.9, 1.0],
            "min_child_weight": [1, 3, 5, 8],
            "gamma": [0.0, 0.1, 0.3],
            "reg_alpha": [0.0, 0.01, 0.1, 0.5],
            "reg_lambda": [0.5, 1.0, 2.0, 5.0],
        }

        if QUICK_MODE:
            parameters = {
                "n_estimators": [80, 120, 180],
                "max_depth": [3, 4, 5],
                "learning_rate": [0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.75, 1.0],
                "min_child_weight": [1, 3],
                "gamma": [0.0, 0.1],
                "reg_alpha": [0.0, 0.1],
                "reg_lambda": [1.0, 2.0],
            }

        n_iter = XGB_SEARCH_ITERATIONS

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return estimator, parameters, n_iter


def train_source_only_model(
    model_name,
    X_source,
    y_source,
    source_groups,
):
    cv = make_spatial_cv(
        y_source,
        source_groups,
    )

    estimator, parameters, n_iter = build_search(
        model_name,
        y_source,
    )

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=parameters,
        n_iter=n_iter,
        scoring="average_precision",
        n_jobs=-1,
        cv=cv,
        random_state=RANDOM_SEED,
        refit=True,
        verbose=1,
    )

    search.fit(
        X_source,
        y_source,
        groups=source_groups,
    )

    # Source-area out-of-fold probabilities only.
    oof_probability = cross_val_predict(
        clone(search.best_estimator_),
        X_source,
        y_source,
        groups=source_groups,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    # Threshold is selected only from source-area OOF predictions.
    threshold = optimal_f1_threshold(
        y_source,
        oof_probability,
    )

    final_model = clone(
        search.best_estimator_
    )
    final_model.fit(
        X_source,
        y_source,
    )

    return (
        final_model,
        threshold,
        search.best_params_,
        float(search.best_score_),
    )


def metric_row(y_true, probability, prediction):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "n": int(len(y_true)),
        "OA": accuracy_score(
            y_true,
            prediction,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "specificity": specificity,
        "F1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "kappa": cohen_kappa_score(
            y_true,
            prediction,
        ),
        "MCC": matthews_corrcoef(
            y_true,
            prediction,
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probability,
        ),
        "PR_AUC": average_precision_score(
            y_true,
            probability,
        ),
        "Brier": brier_score_loss(
            y_true,
            probability,
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def cluster_bootstrap_f1(
    y,
    prediction,
    groups,
    replicates,
    seed,
):
    frame = pd.DataFrame({
        "y": y,
        "prediction": prediction,
        "group": groups,
    }).reset_index(drop=True)

    unique_groups = (
        frame["group"]
        .drop_duplicates()
        .to_numpy()
    )

    rng = np.random.default_rng(seed)
    values = []

    for _ in range(replicates):
        chosen_groups = rng.choice(
            unique_groups,
            size=len(unique_groups),
            replace=True,
        )

        sampled_indices = np.concatenate([
            frame.index[
                frame["group"] == g
            ].to_numpy()
            for g in chosen_groups
        ])

        values.append(
            f1_score(
                frame.loc[
                    sampled_indices,
                    "y",
                ],
                frame.loc[
                    sampled_indices,
                    "prediction",
                ],
                zero_division=0,
            )
        )

    return tuple(
        np.quantile(
            values,
            [0.025, 0.975],
        )
    )


In [ ]:

# CELL 5 — Run BOTH transfer directions automatically

TRANSFER_DIRECTIONS = [
    ("Tanore", "Manda"),
    ("Manda", "Tanore"),
]

result_rows = []
tuning_rows = []
prediction_frames = []

for source_area, target_area in TRANSFER_DIRECTIONS:

    for stream in STREAMS:

        source = (
            sample_tables[source_area][stream]["Training"]
            .copy()
        )

        target = (
            sample_tables[target_area][stream]["Validation"]
            .copy()
        )

        X_source = source[
            FEATURE_NAMES
        ].to_numpy(
            dtype="float32"
        )

        y_source = source[
            "class"
        ].to_numpy(
            dtype=int
        )

        groups_source = source[
            "group"
        ].astype(str).to_numpy()

        X_target = target[
            FEATURE_NAMES
        ].to_numpy(
            dtype="float32"
        )

        y_target = target[
            "class"
        ].to_numpy(
            dtype=int
        )

        target_groups = target[
            "group"
        ].astype(str).to_numpy()

        for model_name in MODELS:

            print("\n" + "=" * 78)
            print(
                f"TRAIN: {source_area}  →  TEST: {target_area}"
            )
            print(
                f"STREAM: {stream} | MODEL: {model_name}"
            )
            print("=" * 78)

            (
                model,
                threshold,
                best_params,
                best_cv_ap,
            ) = train_source_only_model(
                model_name,
                X_source,
                y_source,
                groups_source,
            )

            target_probability = model.predict_proba(
                X_target
            )[:, 1]

            target_prediction = (
                target_probability
                >= threshold
            ).astype(
                "uint8"
            )

            metrics = metric_row(
                y_target,
                target_probability,
                target_prediction,
            )

            ci_low, ci_high = cluster_bootstrap_f1(
                y_target,
                target_prediction,
                target_groups,
                BOOTSTRAP_REPLICATES,
                RANDOM_SEED,
            )

            result_rows.append({
                "source_area": source_area,
                "target_area": target_area,
                "stream": stream,
                "model": model_name,
                "threshold_from_source": threshold,
                "source_cv_average_precision": best_cv_ap,
                "F1_CI_low": ci_low,
                "F1_CI_high": ci_high,
                **metrics,
            })

            tuning_rows.append({
                "source_area": source_area,
                "target_area": target_area,
                "stream": stream,
                "model": model_name,
                "threshold_from_source": threshold,
                "best_cv_average_precision": best_cv_ap,
                "best_parameters": json.dumps(
                    best_params
                ),
            })

            prediction_frames.append(
                pd.DataFrame({
                    "sample_id": target[
                        "sample_id"
                    ].astype(str),
                    "group": target[
                        "group"
                    ].astype(str),
                    "true_class": y_target,
                    "probability": target_probability,
                    "prediction": target_prediction,
                    "source_area": source_area,
                    "target_area": target_area,
                    "stream": stream,
                    "model": model_name,
                })
            )

            print(
                "Target F1:",
                round(metrics["F1"], 4),
                "| MCC:",
                round(metrics["MCC"], 4),
                "| OA:",
                round(metrics["OA"], 4),
                "| threshold:",
                round(threshold, 4),
            )

results = pd.DataFrame(
    result_rows
)

tuning = pd.DataFrame(
    tuning_rows
)

predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

print("\n✅ BOTH TRANSFER DIRECTIONS FINISHED")
display(
    results.sort_values(
        [
            "source_area",
            "stream",
            "model",
        ]
    ).round(4)
)


In [ ]:

# CELL 6 — Paired Fused-Hybrid vs Planet-only comparison on the SAME target samples

comparison_rows = []

for (
    source_area,
    target_area,
    model_name,
), group in predictions.groupby(
    [
        "source_area",
        "target_area",
        "model",
    ]
):

    fused = group[
        group["stream"] == "FusedHybrid"
    ].copy()

    planet = group[
        group["stream"] == "PlanetOnly"
    ].copy()

    merged = fused.merge(
        planet,
        on=[
            "sample_id",
            "true_class",
        ],
        suffixes=(
            "_fused",
            "_planet",
        ),
    )

    correct_fused = (
        merged["prediction_fused"]
        == merged["true_class"]
    )

    correct_planet = (
        merged["prediction_planet"]
        == merged["true_class"]
    )

    fused_only_correct = int(
        (
            correct_fused
            & ~correct_planet
        ).sum()
    )

    planet_only_correct = int(
        (
            ~correct_fused
            & correct_planet
        ).sum()
    )

    discordant = (
        fused_only_correct
        + planet_only_correct
    )

    if discordant > 0:
        p_value = binomtest(
            fused_only_correct,
            discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    else:
        p_value = 1.0

    fused_f1 = f1_score(
        merged["true_class"],
        merged["prediction_fused"],
        zero_division=0,
    )

    planet_f1 = f1_score(
        merged["true_class"],
        merged["prediction_planet"],
        zero_division=0,
    )

    comparison_rows.append({
        "source_area": source_area,
        "target_area": target_area,
        "model": model_name,
        "n_paired_target_samples": len(merged),
        "fused_only_correct": fused_only_correct,
        "planet_only_correct": planet_only_correct,
        "discordant": discordant,
        "mcnemar_exact_p": p_value,
        "fused_F1": fused_f1,
        "planet_F1": planet_f1,
        "delta_F1_fused_minus_planet": (
            fused_f1
            - planet_f1
        ),
    })

comparisons = pd.DataFrame(
    comparison_rows
)

display(
    comparisons.round(4)
)


In [ ]:

# CELL 7 — Publication-ready confusion matrices

for row in results.itertuples(index=False):

    selected = predictions[
        (predictions["source_area"] == row.source_area)
        & (predictions["target_area"] == row.target_area)
        & (predictions["stream"] == row.stream)
        & (predictions["model"] == row.model)
    ]

    cm = confusion_matrix(
        selected["true_class"],
        selected["prediction"],
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(5.2, 4.6)
    )

    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "Non-Boro",
            "Boro",
        ],
    )

    display_cm.plot(
        ax=ax,
        colorbar=False,
    )

    ax.set_title(
        f"{row.source_area} → {row.target_area}\n"
        f"{row.stream} | {row.model}"
    )

    fig.tight_layout()

    filename = (
        f"CM_{row.source_area}_to_{row.target_area}_"
        f"{row.stream}_{row.model}.png"
    )

    fig.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

print("✅ Confusion matrices saved.")


In [ ]:

# CELL 8 — F1 comparison figure

plot_data = results.copy()

plot_data["experiment"] = (
    plot_data["source_area"]
    + "→"
    + plot_data["target_area"]
    + " | "
    + plot_data["stream"]
    + " | "
    + plot_data["model"]
)

plot_data = plot_data.sort_values(
    "F1"
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

ax.barh(
    plot_data["experiment"],
    plot_data["F1"],
)

minimum = float(
    plot_data["F1"].min()
)

ax.set_xlim(
    max(0.0, minimum - 0.08),
    1.01,
)

ax.set_xlabel(
    "Independent target-area F1 score"
)

ax.set_title(
    "Tanore ↔ Manda Cross-Area Transfer Performance"
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "Cross_Area_Transfer_F1.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:

# CELL 9 — Save all publication tables

input_summary.to_csv(
    TABLE_DIR
    / "Cross_Area_Input_Summary.csv",
    index=False,
)

results.to_csv(
    TABLE_DIR
    / "Cross_Area_Transfer_Metrics.csv",
    index=False,
)

tuning.to_csv(
    TABLE_DIR
    / "Cross_Area_Source_Only_Tuning.csv",
    index=False,
)

predictions.to_csv(
    TABLE_DIR
    / "Cross_Area_Transfer_Predictions.csv",
    index=False,
)

comparisons.to_csv(
    TABLE_DIR
    / "Cross_Area_Fused_vs_Planet_McNemar.csv",
    index=False,
)

excel_path = (
    OUTPUT_ROOT
    / "Cross_Area_Transfer_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:

    input_summary.to_excel(
        writer,
        sheet_name="Input_Summary",
        index=False,
    )

    results.to_excel(
        writer,
        sheet_name="Transfer_Metrics",
        index=False,
    )

    tuning.to_excel(
        writer,
        sheet_name="Source_Only_Tuning",
        index=False,
    )

    comparisons.to_excel(
        writer,
        sheet_name="Fused_vs_Planet",
        index=False,
    )

    predictions.to_excel(
        writer,
        sheet_name="Target_Predictions",
        index=False,
    )

print("\n" + "=" * 78)
print("✅ BIDIRECTIONAL CROSS-AREA TRANSFER COMPLETE")
print("=" * 78)
print("Metrics :", TABLE_DIR / "Cross_Area_Transfer_Metrics.csv")
print("Excel   :", excel_path)
print("Figures :", FIGURE_DIR)



## How to interpret the result

### Example 1
If:

- within-area F1 = 1.00
- Tanore → Manda F1 = 0.88

এটা failure না। এর মানে model source area-তে খুব ভালো হলেও অন্য geographical area-তে generalization কমেছে।

### Example 2
If both directions show high F1/MCC
তাহলে geographic transferability-এর evidence শক্তিশালী হবে।

### Publication wording
**“Models were tuned exclusively on the source-area training data using grouped cross-validation and were then evaluated without recalibration on the independent validation samples of the geographically separate target area.”**

### Important
Target-area training samples এই notebook-এর model fitting বা threshold selection-এ ব্যবহার করা হয় না।
